In [ ]:
import random
from collections import defaultdict
from copy import deepcopy

import networkx as nx
from networkx.algorithms.community.quality import modularity
from networkx.utils.mapped_queue import MappedQueue

In [ ]:
def greedy_modularity_communities(
    G,
    weight=None,
    resolution=1,
    cutoff=1,
    best_n=None,
):
    if not G.size():
        return [{n} for n in G], []

    if (cutoff < 1) or (cutoff > G.number_of_nodes()):
        raise ValueError(f"cutoff must be between 1 and {len(G)}. Got {cutoff}.")
    if best_n is not None:
        if (best_n < 1) or (best_n > G.number_of_nodes()):
            raise ValueError(f"best_n must be between 1 and {len(G)}. Got {best_n}.")
        if best_n < cutoff:
            raise ValueError(f"Must have best_n >= cutoff. Got {best_n} < {cutoff}")
        if best_n == 1:
            return [set(G)], []
    else:
        best_n = G.number_of_nodes()
    community_gen = _greedy_modularity_communities_generator(
        G, weight=weight, resolution=resolution
    )
    communities = next(community_gen)
    merge_history = []
    while len(communities) > cutoff:
        try:
            merge_record = next(community_gen)  # yields a dict with dq + merge info
        # StopIteration occurs when communities are the connected components
        except StopIteration:
            communities = sorted(communities, key=len, reverse=True)
            # if best_n requires more merging, merge big sets for highest modularity
            step = len(merge_history) + 1
            while len(communities) > best_n:
                comm1, comm2, *rest = communities
                merge_history.append({
                    'step': step,
                    'merged_into': frozenset(comm1 | comm2),
                    'absorbed': frozenset(comm2),
                    'dq': None,  # forced merge, no dq available
                    'edges': None,
                    'num_communities_after': len(rest) + 1,
                })
                communities = [comm1 ^ comm2]
                communities.extend(rest)
                step += 1
            return communities, merge_history

        # merge_record is a dict yielded from the generator
        dq = merge_record['dq']

        # keep going unless max_mod is reached or best_n says to merge more
        if dq < 0 and len(communities) <= best_n:
            break

        communities = next(community_gen)  # yields updated communities

        # record this merge step
        merge_history.append({
            'step': len(merge_history) + 1,
            'merged_into': merge_record['v'],
            'absorbed':    merge_record['u'],
            'dq':          merge_record['dq'],
            'edges':       merge_record['edges'],
            'num_communities_after': len(communities),
        })

    return sorted(communities, key=len, reverse=True), merge_history


In [ ]:
def _greedy_modularity_communities_generator(G, weight=None, resolution=1):
    directed = G.is_directed()
    N = G.number_of_nodes()

    # Count edges (or the sum of edge-weights for weighted graphs)
    m = G.size(weight)
    q0 = 1 / m

    # Calculate degrees (notation from the papers)
    # a : the fraction of (weighted) out-degree for each node
    # b : the fraction of (weighted) in-degree for each node
    if directed:
        a = {node: deg_out * q0 for node, deg_out in G.out_degree(weight=weight)}
        b = {node: deg_in * q0 for node, deg_in in G.in_degree(weight=weight)}
    else:
        a = b = {node: deg * q0 * 0.5 for node, deg in G.degree(weight=weight)}

    # this preliminary step collects the edge weights for each node pair
    dq_dict = defaultdict(lambda: defaultdict(float))
    for u, v, wt in G.edges(data=weight, default=1):
        if u == v:
            continue
        dq_dict[u][v] += wt
        dq_dict[v][u] += wt

    # now scale and subtract the expected edge-weights term
    for u, nbrdict in dq_dict.items():
        for v, wt in nbrdict.items():
            dq_dict[u][v] = q0 * wt - resolution * (a[u] * b[v] + b[u] * a[v])

    # Use -dq to get a max_heap instead of a min_heap
    dq_heap = {u: MappedQueue({(u, v): -dq for v, dq in dq_dict[u].items()}) for u in G}
    # H -> all_dq_heap holds a heap with the best items for each node
    H = MappedQueue([dq_heap[n].heap[0] for n in G if len(dq_heap[n]) > 0])

    # Initialize single-node communities
    communities = {n: frozenset([n]) for n in G}
    yield communities.values()

    # Merge the two communities that lead to the largest modularity
    while len(H) > 1:
        # Find best merge
        try:
            negdq, u, v = H.pop()
        except IndexError:
            break
        dq = -negdq

        # Compute cross-edges between the two communities being merged
        merge_edges = [
            (x, y) for x in communities[u]
                   for y in communities[v]
                   if G.has_edge(x, y)
        ]

        # Yield merge info dict BEFORE applying the merge
        # (caller reads dq and edge info, then calls next() to get updated communities)
        yield {'u': u, 'v': v, 'dq': dq, 'edges': merge_edges}

        # Remove best merge from row u heap
        dq_heap[u].pop()
        # Push new row max onto H
        if len(dq_heap[u]) > 0:
            H.push(dq_heap[u].heap[0])
        # If this element was also at the root of row v, remove the duplicate from H
        if dq_heap[v].heap[0] == (v, u):
            H.remove((v, u))
            dq_heap[v].remove((v, u))
            if len(dq_heap[v]) > 0:
                H.push(dq_heap[v].heap[0])
        else:
            dq_heap[v].remove((v, u))

        # Perform merge: absorb u into v
        communities[v] = frozenset(communities[u] | communities[v])
        del communities[u]

        # Get neighbour communities connected to the merged communities
        u_nbrs = set(dq_dict[u])
        v_nbrs = set(dq_dict[v])
        all_nbrs = (u_nbrs | v_nbrs) - {u, v}
        both_nbrs = u_nbrs & v_nbrs
        # Update dq for merge of u into v
        for w in all_nbrs:
            if w in both_nbrs:
                dq_vw = dq_dict[v][w] + dq_dict[u][w]
            elif w in v_nbrs:
                dq_vw = dq_dict[v][w] - resolution * (a[u] * b[w] + a[w] * b[u])
            else:  # w in u_nbrs
                dq_vw = dq_dict[u][w] - resolution * (a[v] * b[w] + a[w] * b[v])
            for row, col in [(v, w), (w, v)]:
                dq_heap_row = dq_heap[row]
                dq_dict[row][col] = dq_vw
                if len(dq_heap_row) > 0:
                    d_oldmax = dq_heap_row.heap[0]
                else:
                    d_oldmax = None
                d = (row, col)
                d_negdq = -dq_vw
                if w in v_nbrs:
                    dq_heap_row.update(d, d, priority=d_negdq)
                else:
                    dq_heap_row.push(d, priority=d_negdq)
                if d_oldmax is None:
                    H.push(d, priority=d_negdq)
                else:
                    row_max = dq_heap_row.heap[0]
                    if d_oldmax != row_max or d_oldmax.priority != row_max.priority:
                        H.update(d_oldmax, row_max)

        # Remove row/col u from dq_dict matrix
        for w in dq_dict[u]:
            dq_old = dq_dict[w][u]
            del dq_dict[w][u]
            if w != v:
                for row, col in [(w, u), (u, w)]:
                    dq_heap_row = dq_heap[row]
                    d_old = (row, col)
                    if dq_heap_row.heap[0] == d_old:
                        dq_heap_row.remove(d_old)
                        H.remove(d_old)
                        if len(dq_heap_row) > 0:
                            H.push(dq_heap_row.heap[0])
                    else:
                        dq_heap_row.remove(d_old)

        del dq_dict[u]
        dq_heap[u] = MappedQueue()
        # Merge u into v and update a
        a[v] += a[u]
        a[u] = 0
        if directed:
            b[v] += b[u]
            b[u] = 0

        yield communities.values()


In [ ]:
communities, merge_history = greedy_modularity_communities(net.get_nx().copy(), cutoff=5)

print(f"Final communities ({len(communities)} total):")
for i, c in enumerate(communities):
    print(f"  Community {i}: {sorted(c)}")

print(f"\nMerge history ({len(merge_history)} steps):")
for record in merge_history:
    print(
        f"  Step {record['step']:>3}: "
        f"absorbed {record['absorbed']} into {record['merged_into']}  "
        f"dq={record['dq']:+.6f}  "
        f"cross-edges={record['edges']}  "
        f"communities_after={record['num_communities_after']}"
    )
merge_history

In [ ]:
def get_seed_set_from_merges(merge_history, k, deduplicate=True):
    last_k = merge_history[-k:]  # the final k merges in order
    seeds = []
    for record in last_k:
        if record['edges']:
            for u, v in record['edges']:
                seeds.append(u)
                seeds.append(v)
    return list(set(seeds))

In [ ]:
def get_seed_set_by_degree(G, merge_history, k, n_seeds=None):
    last_k = merge_history[-k:]
    # collect all endpoint nodes from cross-edges in the last k merges
    candidates = set()
    for record in merge_history:
        if record['edges']:
            for u, v in record['edges']:
                candidates.add(u)
                candidates.add(v)
    ranked = sorted(candidates, key=lambda node: G.degree[node], reverse=True)
    return ranked[:n_seeds] if n_seeds is not None else ranked

In [ ]:
seed_set = get_seed_set_from_merges(merge_history, 5, True)
print(list(seed_set))

In [ ]:
from Influence_Modelling import XsvNetwork, Network, Sampling, Metrics, Heuristics
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import scienceplots 
plt.style.use(["science", "no-latex"])
import time

enron  =XsvNetwork.XsvNetwork("enron")
net = Network.Network("")
net.set_nx(Sampling.mhda_sample(enron.get_nx(), 1000))

net2 = Network.Network("")
net2.set_nx(net.get_nx().copy())

communities, merge_history = greedy_modularity_communities(net.get_nx().copy(), cutoff=5)

deg = Heuristics.deg_h(net.get_nx(), 10)
deg = net.run_ICC(deg)

seed_set = get_seed_set_by_degree(net2.get_nx(), merge_history, 50, 10)
community = net2.run_ICC(list(seed_set))

In [ ]:
print(community[-1])
print(deg[-1])

In [ ]:
sizes = [11]

deg = [net.run_ICC(Heuristics.double_deg_discount_h(net.get_nx(), size)) for size in sizes]
deg = [de[2][-1] for de in deg]

bet = [net.run_ICC(Heuristics.local_clustering_coefficient_h(net.get_nx(), size)) for size in sizes]
bet = [de[2][-1] for de in bet]

ties_res = [net.run_ICC(list(ties), size) for size in sizes]
ties_res = [de[2][-1] for de in ties_res]

fig, ax = plt.subplots()
fig.set_size_inches(5,4)
fig.set_dpi(200)
ax.plot(sizes, deg, label="Double Discount", marker="^")
ax.plot(sizes, bet, label="Clustering Coef.", marker="o")
ax.plot(sizes, ties_res, label="Ties", marker="*")
ax.set_xlabel("Seed Nodes")
ax.set_ylabel("Final Influence")
ax.set_title("100 node enron MHDA")
ax.legend()

In [ ]:
import itertools
G = nx.erdos_renyi_graph(100, 0.2)
com = nx.community.girvan_newman(G)
for communities in itertools.islice(com, 20):
    print(tuple(sorted(c) for c in communities))